In [1]:
# ==============================================================================
# ステップ 1：OOD不確実性解析（数値解析CSV ＋ グラフPDF出力版）
# ==============================================================================
library(earth)
library(ggplot2)
library(dplyr)

# 1. パスの設定
dir_path <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/OOD_check/gcvEarth/"
file_name <- "fixed20250702_model_data_gcvEarth_n_all_FP_PCEmax_20251212.rds"
target_path <- paste0(dir_path, file_name)
today <- format(Sys.Date(), "%Y%m%d")

# ロード
bundle <- readRDS(target_path)
model    <- bundle$model
train_df <- bundle$training_data
ood_df   <- bundle$ood_test_data

# 2. 予測と不確実性の算出
preds   <- predict(model, newdata = ood_df)
actuals <- ood_df$PCEmax
errors  <- abs(as.numeric(preds) - as.numeric(actuals))

# 3. 構造的距離（未知度）の算出
used_features <- model$finalModel$namesx
train_feat <- train_df[, used_features, drop = FALSE]
ood_feat   <- ood_df[, used_features, drop = FALSE]
center    <- colMeans(train_feat)
distances <- apply(ood_feat, 1, function(x) sqrt(sum((x - center)^2)))

# 4. データの統合
df_viz <- data.frame(
  Sample_ID = rownames(ood_df),
  Distance  = distances,
  Predicted = as.numeric(preds),
  Actual    = actuals,
  Error     = errors
)

# --- 5. 数値レポートの作成と保存 ---
# 四分位数グループ集計
df_summary <- df_viz %>%
  mutate(Dist_Group = ntile(Distance, 4)) %>%
  group_by(Dist_Group) %>%
  summarise(
    Avg_Distance = mean(Distance),
    Avg_Error    = mean(Error),
    Max_Error    = max(Error),
    Count        = n(),
    .groups = 'drop'
  )

# CSV出力（集計結果と生データの両方）
csv_summary_file <- paste0("step1_ood_uncertainty_summary_", today, ".csv")
csv_raw_file     <- paste0("step1_ood_uncertainty_raw_data_", today, ".csv")
write.csv(df_summary, csv_summary_file, row.names = FALSE)
write.csv(df_viz, csv_raw_file, row.names = FALSE)

# --- 6. グラフ作成とPDF保存 ---
p1 <- ggplot(df_viz, aes(x = Distance, y = Predicted)) +
  geom_errorbar(aes(ymin = Predicted - Error, ymax = Predicted + Error), 
                width = 0.5, color = "gray70", alpha = 0.6) +
  geom_point(aes(color = Error), size = 3) +
  scale_color_gradient(low = "blue", high = "red") +
  theme_minimal(base_size = 14) +
  labs(title = "Fixed Model: n_all_FP (PCEmax) OOD Uncertainty",
       x = "Structural Distance (Unseen Degree)",
       y = "Predicted PCEmax [%]",
       color = "Abs Error")

pdf_file <- paste0("step1_ood_uncertainty_plot_", today, ".pdf")
ggsave(pdf_file, p1, width = 8, height = 6)

# --- コンソールへの最終レポート ---
cat("====================================================\n")
cat("  ステップ 1：解析完了レポート\n")
cat("====================================================\n")
print(df_summary)
cat("\n--- 出力ファイル一覧 ---\n")
cat("1. 集計CSV:", csv_summary_file, "\n")
cat("2. 生データCSV:", csv_raw_file, "\n")
cat("3. グラフPDF:", pdf_file, "\n")
cat("----------------------------------------------------\n")
cat("距離と誤差の相関係数:", round(cor(df_viz$Distance, df_viz$Error), 3), "\n")
cat("====================================================\n")

Loading required package: Formula

Loading required package: plotmo

Loading required package: plotrix


Attaching package: 'dplyr'


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union




  <U+30B9><U+30C6><U+30C3><U+30D7> 1<U+FF1A><U+89E3><U+6790><U+5B8C><U+4E86><U+30EC><U+30DD><U+30FC><U+30C8>
# A tibble: 4 x 5
  Dist_Group Avg_Distance Avg_Error Max_Error Count
       <int>        <dbl>     <dbl>     <dbl> <int>
1          1         13.0      0.49     0.611     2
2          2         13.4      1.04     1.72      2
3          3         13.7      2.00     3.29      2
4          4         13.7      3.29     3.29      1

--- <U+51FA><U+529B><U+30D5><U+30A1><U+30A4><U+30EB><U+4E00><U+89A7> ---
1. <U+96C6><U+8A08>CSV: step1_ood_uncertainty_summary_20251212.csv 
2. <U+751F><U+30C7><U+30FC><U+30BF>CSV: step1_ood_uncertainty_raw_data_20251212.csv 
3. <U+30B0><U+30E9><U+30D5>PDF: step1_ood_uncertainty_plot_20251212.pdf 
----------------------------------------------------
<U+8DDD><U+96E2><U+3068><U+8AA4><U+5DEE><U+306E><U+76F8><U+95A2><U+4FC2><U+6570>: 0.733 
